# Task 3 — Numerical Outlier Detection and Treatment
## Women's Clothing: Trends and Analysis in the Fashion Industry

**Pipeline position:** Task 2 standardized interim dataset → final processed dataset.

This notebook compares three detection approaches:
1. IQR fences (univariate distribution-based detection)
2. Z-score (standard-deviation distance)
3. DBSCAN (density-based multivariate anomaly detection)

The notebook does not automatically delete every flagged row. It uses conservative, configurable treatment rules: cap skew-prone business measures, preserve rating scales, and exclude identifier/count-like fields from inappropriate transformations. Review the flagged records and treatment summary before using the final data.

**Input:** `data/interim/womens_clothing_standardized.csv`  
**Output:** `data/processed/womens_clothing_final.csv` and outlier reports.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "data" / "interim" / "womens_clothing_standardized.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "womens_clothing_final.csv"
REPORT_DIR = ROOT / "reports"
LOG_DIR = ROOT / "logs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError("Run Task_2_Standardization.ipynb first. Expected: " + str(INPUT_PATH))

df = pd.read_csv(INPUT_PATH, low_memory=False)
print("Loaded Task 2 output:", df.shape)
display(df.head())

Loaded Task 2 output: (1539, 40)


,title,title_orig,price,retail_price,currency_buyer,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,...,merchant_name,merchant_info_subtitle,merchant_rating_count,merchant_rating,merchant_id,merchant_has_profile_picture,merchant_profile_picture,product_url,product_picture,product_id
0,2020 Summer Vintage Flamingo Print Pajamas Se...,2020 Summer Vintage Flamingo Print Pajamas Se...,16.00,14,EUR,100,False,3.76,54,26.0,...,zgrdejia,(568 notes),568,4.128521,595097d6a26f6e070cb878d1,False,NaN,https://www.wish.com/c/5e9ae51d43d6a96e303acdb0,https://contestimg.wish.com/api/webimage/5e9ae...,5e9ae51d43d6a96e303acdb0
1,SSHOUSE Summer Casual Sleeveless Soirée Party ...,Women's Casual Summer Sleeveless Sexy Mini Dress,8.00,22,EUR,20000,True,3.45,6135,2269.0,...,sarahouse,"83 % avis positifs (17,752 notes)",17752,3.899673,56458aa03a698c35c9050988,False,NaN,https://www.wish.com/c/58940d436a0d3d5da4e95a38,https://contestimg.wish.com/api/webimage/58940...,58940d436a0d3d5da4e95a38
2,2020 Nouvelle Arrivée Femmes Printemps et Été ...,2020 New Arrival Women Spring and Summer Beach...,8.00,43,EUR,100,False,3.57,14,5.0,...,hxt520,86 % avis positifs (295 notes),295,3.989831,5d464a1ffdf7bc44ee933c65,False,NaN,https://www.wish.com/c/5ea10e2c617580260d55310a,https://contestimg.wish.com/api/webimage/5ea10...,5ea10e2c617580260d55310a
3,Hot Summer Cool T-shirt pour les femmes Mode T...,Hot Summer Cool T Shirt for Women Fashion Tops...,8.00,8,EUR,5000,True,4.03,579,295.0,...,allenfan,"(23,832 notes)",23832,4.020435,58cfdefdacb37b556efdff7c,False,NaN,https://www.wish.com/c/5cedf17ad1d44c52c59e4aca,https://contestimg.wish.com/api/webimage/5cedf...,5cedf17ad1d44c52c59e4aca
4,Femmes Shorts d'été à lacets taille élastique ...,Women Summer Shorts Lace Up Elastic Waistband ...,2.72,3,EUR,100,True,3.10,20,6.0,...,happyhorses,"85 % avis positifs (14,482 notes)",14482,4.001588,5ab3b592c3911a095ad5dadb,False,NaN,https://www.wish.com/c/5ebf5819ebac372b070b0e70,https://contestimg.wish.com/api/webimage/5ebf5...,5ebf5819ebac372b070b0e70


## 1. Select appropriate numeric features

Identifiers are excluded. The rating field is bounded by its domain (normally 1–5), so it is validated separately rather than treated like an unrestricted continuous variable.

In [2]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
identifier_keywords = ("id", "url", "picture")
candidate_cols = [
    c for c in numeric_cols
    if not any(k in c for k in identifier_keywords)
    and df[c].nunique(dropna=True) > 2
]
# Keep only columns with enough non-missing observations.
analysis_cols = [c for c in candidate_cols if df[c].notna().sum() >= 5]
print("Numeric columns:", numeric_cols)
print("Outlier analysis columns:", analysis_cols)

if not analysis_cols:
    raise ValueError("No suitable numeric columns available for outlier analysis.")

Numeric columns: ['price', 'retail_price', 'units_sold', 'rating', 'rating_count', 'rating_five_count', 'rating_four_count', 'rating_three_count', 'rating_two_count', 'rating_one_count', 'badges_count', 'product_variation_inventory', 'shipping_option_price', 'countries_shipped_to', 'inventory_total', 'merchant_rating_count', 'merchant_rating']
Outlier analysis columns: ['price', 'retail_price', 'units_sold', 'rating', 'rating_count', 'rating_five_count', 'rating_four_count', 'rating_three_count', 'rating_two_count', 'rating_one_count', 'badges_count', 'product_variation_inventory', 'shipping_option_price', 'countries_shipped_to', 'inventory_total', 'merchant_rating_count', 'merchant_rating']


## 2. IQR detection

The IQR method flags values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR. It is applied per feature and records lower/upper bounds.

In [3]:
iqr_flags = pd.DataFrame(False, index=df.index, columns=analysis_cols)
iqr_bounds = {}
for col in analysis_cols:
    series = df[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_flags[col] = df[col].notna() & ((df[col] < lower) | (df[col] > upper))
    iqr_bounds[col] = {"q1": float(q1), "q3": float(q3), "iqr": float(iqr),
                       "lower": float(lower), "upper": float(upper),
                       "flagged": int(iqr_flags[col].sum())}
display(pd.DataFrame(iqr_bounds).T)

,q1,q3,iqr,lower,upper,flagged
price,5.825000,11.00000,5.175000,-1.937500,18.762500,19.0
retail_price,7.000000,26.00000,19.000000,-21.500000,54.500000,205.0
units_sold,100.000000,5000.00000,4900.000000,-7250.000000,12350.000000,126.0
rating,3.550000,4.11000,0.560000,2.710000,4.950000,95.0
rating_count,26.000000,873.00000,847.000000,-1244.500000,2143.500000,179.0
rating_five_count,13.000000,431.00000,418.000000,-614.000000,1058.000000,176.0
rating_four_count,5.000000,171.00000,166.000000,-244.000000,420.000000,179.0
rating_three_count,4.000000,134.00000,130.000000,-191.000000,329.000000,162.0
rating_two_count,2.000000,64.00000,62.000000,-91.000000,157.000000,142.0
rating_one_count,4.000000,97.00000,93.000000,-135.500000,236.500000,155.0


## 3. Z-score detection

Z-scores measure distance from the feature mean in standard deviations. A threshold of 3 is used as a diagnostic, not as an automatic deletion rule. Strongly skewed features may be better described by IQR.

In [4]:
z_flags = pd.DataFrame(False, index=df.index, columns=analysis_cols)
z_summary = {}
for col in analysis_cols:
    mean = df[col].mean()
    std = df[col].std()
    if pd.notna(std) and std > 0:
        z = (df[col] - mean) / std
        z_flags[col] = z.abs() > 3
    z_summary[col] = int(z_flags[col].sum())

print("Z-score outlier counts (|z| > 3):")
display(pd.Series(z_summary, name="flagged_rows").to_frame())

Z-score outlier counts (|z| > 3):


,flagged_rows
price,9
retail_price,32
units_sold,23
rating,14
rating_count,38
rating_five_count,35
rating_four_count,39
rating_three_count,42
rating_two_count,34
rating_one_count,35


## 4. DBSCAN density-based detection

DBSCAN is applied to standardized numeric features. Missing values are median-imputed for this diagnostic only. Rows labeled `-1` are density-based noise candidates, not guaranteed data errors. The sample-size guard avoids excessive memory use on very large datasets.

In [5]:
DBSCAN_MAX_ROWS = 20000
dbscan_features = [c for c in analysis_cols if df[c].nunique(dropna=True) > 1]
dbscan_labels = pd.Series(index=df.index, data=np.nan, dtype="float")
dbscan_status = "not_run"

if len(df) >= 5 and dbscan_features:
    sample_df = df[dbscan_features].copy()
    sample_df = sample_df.replace([np.inf, -np.inf], np.nan)
    sample_df = sample_df.fillna(sample_df.median())
    if len(sample_df) > DBSCAN_MAX_ROWS:
        sample_df = sample_df.sample(DBSCAN_MAX_ROWS, random_state=42)
    scaled = StandardScaler().fit_transform(sample_df)
    labels = DBSCAN(eps=1.5, min_samples=5).fit_predict(scaled)
    dbscan_labels.loc[sample_df.index] = labels
    dbscan_status = "run_on_sample" if len(sample_df) < len(df) else "run_on_full_data"

dbscan_noise = dbscan_labels.eq(-1).fillna(False)
print("DBSCAN status:", dbscan_status)
print("DBSCAN noise candidates:", int(dbscan_noise.sum()))

DBSCAN status: run_on_full_data
DBSCAN noise candidates: 364


## 5. Domain-aware treatment decisions

Default policy:
- `price`, `retail_price`, and `shipping_option_price`: cap extreme values at IQR fences; these are non-negative monetary measures where extreme values may be valid but can dominate analysis.
- `rating`: validate against the expected 1–5 scale; do not winsorize valid ratings.
- Count and inventory fields: cap only at the IQR upper/lower fence for this baseline analysis, preserving rows.
- Other numeric fields: detect and report, but do not alter automatically.

You can edit `TREATMENT_POLICY` after inspecting distributions and product-domain context.

In [6]:
TREATMENT_POLICY = {
    "price": "cap_iqr",
    "retail_price": "cap_iqr",
    "shipping_option_price": "cap_iqr",
    "units_sold": "cap_iqr",
    "rating_count": "cap_iqr",
    "rating_five_count": "cap_iqr",
    "rating_four_count": "cap_iqr",
    "rating_three_count": "cap_iqr",
    "rating_two_count": "cap_iqr",
    "rating_one_count": "cap_iqr",
    "badges_count": "cap_iqr",
    "product_variation_inventory": "cap_iqr",
    "countries_shipped_to": "cap_iqr",
    "inventory_total": "cap_iqr",
    "merchant_rating_count": "cap_iqr"
}

df_treated = df.copy()
treatment_log = {}
for col, bounds in iqr_bounds.items():
    policy = TREATMENT_POLICY.get(col, "report_only")
    before = df_treated[col].copy()
    if policy == "cap_iqr":
        df_treated[col] = df_treated[col].clip(lower=bounds["lower"], upper=bounds["upper"])
    elif policy == "log1p" and (df_treated[col].dropna() >= 0).all():
        df_treated[col] = np.log1p(df_treated[col])
    elif policy == "remove_rows":
        # Explicitly opt-in only; avoid default row deletion.
        mask = iqr_flags[col]
        df_treated = df_treated.loc[~mask].copy()
    treatment_log[col] = {
        "method": policy,
        "iqr_flagged": bounds["flagged"],
        "values_changed": int((before != df_treated.loc[before.index, col]).fillna(False).sum())
            if col in df_treated.columns else 0
    }

# Validate rating domain without changing valid observations.
if "rating" in df_treated.columns:
    invalid_rating = df_treated["rating"].notna() & ~df_treated["rating"].between(1, 5)
    treatment_log["rating_domain_validation"] = {
        "expected_range": [1, 5],
        "invalid_count": int(invalid_rating.sum())
    }

print("Treatment summary:")
display(pd.DataFrame(treatment_log).T)

Treatment summary:


,method,iqr_flagged,values_changed,expected_range,invalid_count
price,cap_iqr,19,19,NaN,NaN
retail_price,cap_iqr,205,205,NaN,NaN
units_sold,cap_iqr,126,126,NaN,NaN
rating,report_only,95,0,NaN,NaN
rating_count,cap_iqr,179,179,NaN,NaN
rating_five_count,cap_iqr,176,219,NaN,NaN
rating_four_count,cap_iqr,179,222,NaN,NaN
rating_three_count,cap_iqr,162,205,NaN,NaN
rating_two_count,cap_iqr,142,185,NaN,NaN
rating_one_count,cap_iqr,155,198,NaN,NaN


## 6. Save detection reports and final dataset

The report includes IQR counts, Z-score counts, DBSCAN noise status, and selected treatments. Detection flags are also saved separately so the decisions remain auditable.

In [7]:
flag_report = pd.DataFrame({
    "iqr_any_feature": iqr_flags.any(axis=1),
    "zscore_any_feature": z_flags.any(axis=1),
    "dbscan_noise_candidate": dbscan_noise
}, index=df.index)
flag_report.to_csv(REPORT_DIR / "task3_row_outlier_flags.csv", index_label="row_index")

feature_report = pd.DataFrame({
    "iqr_flagged": iqr_flags.sum(),
    "zscore_flagged": z_flags.sum()
})
feature_report["dbscan_noise_rows_dataset"] = int(dbscan_noise.sum())
feature_report.to_csv(REPORT_DIR / "task3_feature_outlier_summary.csv", index_label="feature")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_treated.to_csv(OUTPUT_PATH, index=False)

summary = {
    "input_rows": int(len(df)),
    "output_rows": int(len(df_treated)),
    "input_columns": int(len(df.columns)),
    "output_columns": int(len(df_treated.columns)),
    "dbscan_status": dbscan_status,
    "dbscan_noise_candidates": int(dbscan_noise.sum()),
    "treatments": treatment_log
}
with open(LOG_DIR / "task3_treatment_log.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, default=str)

print("Task 3 complete. Final dataset:", OUTPUT_PATH)
print("Final shape:", df_treated.shape)
display(df_treated.head())

Task 3 complete. Final dataset: d:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project\data\processed\womens_clothing_final.csv
Final shape: (1539, 40)


,title,title_orig,price,retail_price,currency_buyer,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,...,merchant_name,merchant_info_subtitle,merchant_rating_count,merchant_rating,merchant_id,merchant_has_profile_picture,merchant_profile_picture,product_url,product_picture,product_id
0,2020 Summer Vintage Flamingo Print Pajamas Se...,2020 Summer Vintage Flamingo Print Pajamas Se...,16.00,14.0,EUR,100,False,3.76,54.0,26.0,...,zgrdejia,(568 notes),568.0,4.128521,595097d6a26f6e070cb878d1,False,NaN,https://www.wish.com/c/5e9ae51d43d6a96e303acdb0,https://contestimg.wish.com/api/webimage/5e9ae...,5e9ae51d43d6a96e303acdb0
1,SSHOUSE Summer Casual Sleeveless Soirée Party ...,Women's Casual Summer Sleeveless Sexy Mini Dress,8.00,22.0,EUR,12350,True,3.45,2143.5,1058.0,...,sarahouse,"83 % avis positifs (17,752 notes)",17752.0,3.899673,56458aa03a698c35c9050988,False,NaN,https://www.wish.com/c/58940d436a0d3d5da4e95a38,https://contestimg.wish.com/api/webimage/58940...,58940d436a0d3d5da4e95a38
2,2020 Nouvelle Arrivée Femmes Printemps et Été ...,2020 New Arrival Women Spring and Summer Beach...,8.00,43.0,EUR,100,False,3.57,14.0,5.0,...,hxt520,86 % avis positifs (295 notes),295.0,3.989831,5d464a1ffdf7bc44ee933c65,False,NaN,https://www.wish.com/c/5ea10e2c617580260d55310a,https://contestimg.wish.com/api/webimage/5ea10...,5ea10e2c617580260d55310a
3,Hot Summer Cool T-shirt pour les femmes Mode T...,Hot Summer Cool T Shirt for Women Fashion Tops...,8.00,8.0,EUR,5000,True,4.03,579.0,295.0,...,allenfan,"(23,832 notes)",23832.0,4.020435,58cfdefdacb37b556efdff7c,False,NaN,https://www.wish.com/c/5cedf17ad1d44c52c59e4aca,https://contestimg.wish.com/api/webimage/5cedf...,5cedf17ad1d44c52c59e4aca
4,Femmes Shorts d'été à lacets taille élastique ...,Women Summer Shorts Lace Up Elastic Waistband ...,2.72,3.0,EUR,100,True,3.10,20.0,6.0,...,happyhorses,"85 % avis positifs (14,482 notes)",14482.0,4.001588,5ab3b592c3911a095ad5dadb,False,NaN,https://www.wish.com/c/5ebf5819ebac372b070b0e70,https://contestimg.wish.com/api/webimage/5ebf5...,5ebf5819ebac372b070b0e70


### Task 3 outcome
- IQR, Z-score, and DBSCAN are used as complementary detection methods.
- Outlier flags are reported separately from treatment decisions.
- Selected business measures are capped rather than deleting entire records.
- Rating values are checked against their expected scale.
- The final processed dataset and audit reports are saved.